# Hourly Delay Profile (Polars)

Shows the local hours when buses are most late using the Polars analysis path.

In [ ]:
from pathlib import Path
import importlib.util
import sys

import plotly.express as px
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "analysis" / "polars").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from the project root or a subdirectory.")

POLARS_ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "polars"
polars_analysis_path = str(POLARS_ANALYSIS_DIR)
if polars_analysis_path in sys.path:
    sys.path.remove(polars_analysis_path)
sys.path.insert(0, polars_analysis_path)

for module_name in ("_shared", "report_cache", "cli_common"):
    module = sys.modules.get(module_name)
    module_file = getattr(module, "__file__", None) if module else None
    module_path = Path(module_file).resolve() if module_file else None
    if module_path is not None and POLARS_ANALYSIS_DIR not in module_path.parents:
        sys.modules.pop(module_name, None)

def load_polars_script(module_name: str, filename: str):
    spec = importlib.util.spec_from_file_location(module_name, POLARS_ANALYSIS_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

hourly = load_polars_script("polars_hourly_delay_profile", "hourly-delay-profile.py")

DB = PROJECT_ROOT / "data" / "foli.db"
CACHE_DIR = PROJECT_ROOT / "outputs" / "polars-report-cache"
FORCE_CACHE = False
NO_CACHE = False
TIMEZONE = "Europe/Helsinki"
LINE_REF = None
MIN_OBSERVATIONS = 30
LIMIT = 24
QUALITY_MODE = "conservative"
BUCKET = "trip-stop"


Set `NO_CACHE = True` to read SQLite directly, or `FORCE_CACHE = True` to rebuild the Polars cache.

In [ ]:
class Args:
    db = DB
    cache_dir = CACHE_DIR
    force_cache = FORCE_CACHE
    no_cache = NO_CACHE
    timezone = TIMEZONE
    line_ref = LINE_REF
    min_observations = MIN_OBSERVATIONS
    limit = LIMIT
    quality_mode = QUALITY_MODE
    exclude_stop_call_disagreement = False
    bucket = BUCKET

buckets = hourly.load_delay_buckets_for_args(Args)
profile = hourly.build_hourly_delay_profile(
    buckets,
    line_ref=LINE_REF,
    min_observations=MIN_OBSERVATIONS,
    limit=LIMIT,
)
profile


Delay has positive and negative values. Positive values mean the bus is late; negative values mean the bus is early.

In [ ]:
if profile.is_empty():
    print("No matching observations found.")
else:
    fig = px.bar(
        profile.sort("hour_local"),
        x="hour_local",
        y="p90_delay_min",
        title="P90 delay by local hour",
        labels={"hour_local": "Local hour", "p90_delay_min": "P90 delay, minutes"},
    )
    fig.update_layout(showlegend=False)
    fig.show()
